# **The Optimized SVM + TF-IDF Cell**

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from skopt import BayesSearchCV
from skopt.space import Real, Categorical

# 1. Feature Generation (Lemmatized Process)
print("⚙️ Generating TF-IDF Features for Lemmatized Process...")
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X = tfidf_vectorizer.fit_transform(df_sentiment['Text_Lemmatized'])
y = df_sentiment['Majority_Truth'].values

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"✅ Data Split: Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}")

# 3. Define the Bayesian Search Space
# This hits your rubric requirement to handle over/underfitting via complexity/regularization
search_spaces = {
    'C': Real(0.1, 50.0, prior='log-uniform'),      # Regularization parameter
    'kernel': Categorical(['linear', 'rbf']),       # Testing approach changes (Linear -> RBF)
    'gamma': Real(1e-3, 1.0, prior='log-uniform')   # Kernel coefficient (Only impacts RBF)
}

# 4. Initialize the Optimizer
print("\n🚀 Setting up Bayesian Optimization...")
# Note: probability=True is required to calculate ROC-AUC later!
svm_base = SVC(probability=True, random_state=42)

# We use 10 iterations to save time. In a real corporate setting, you might run 50+.
n_iterations = 10
bayes_search = BayesSearchCV(
    estimator=svm_base,
    search_spaces=search_spaces,
    n_iter=n_iterations,
    cv=3,                 # 3-Fold Cross Validation
    scoring='f1_macro',   # Optimizing specifically for F1-Score (per your rubric)
    n_jobs=-1,            # Use all CPU cores
    random_state=42
)

# Custom Callback to make skopt work with tqdm
class TqdmSkoptCallback:
    def __init__(self, n_iter):
        self.pbar = tqdm(total=n_iter, desc="Bayesian Tuning", bar_format='{l_bar}{bar}| {elapsed} [{remaining} left]')
    def __call__(self, res):
        self.pbar.update(1)

# 5. Execute the Search
print(f"🧠 Commencing Search (Testing {n_iterations} different model architectures...)")
# This step takes time! The model is training itself multiple times.
bayes_search.fit(X_train, y_train, callback=TqdmSkoptCallback(n_iterations))

# 6. Extract the Best Model & Predict
best_svm = bayes_search.best_estimator_
print("\n🎯 Generating Predictions with the Champion Model...")
y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:, 1] # Get probabilities for ROC-AUC

# 7. Calculate Rubric Metrics
opt_acc = accuracy_score(y_test, y_pred)
opt_roc_auc = roc_auc_score(y_test, y_prob)

print("\n" + "═"*60)
print(f" OPTIMIZED SVM EXPERIMENT: TF-IDF")
print("═"*60)
print("BEST HYPERPARAMETERS FOUND:")
for param, value in bayes_search.best_params_.items():
    print(f" - {param}: {value}")
print("-" * 30)
print(f"Final Accuracy: {opt_acc * 100:.2f}%")
print(f"ROC-AUC Score:  {opt_roc_auc:.4f}")
print("\nDetailed Classification Report (includes Precision/Recall/F1):")
print("-" * 30)
print(classification_report(y_test, y_pred, target_names=['Negative (0)', 'Positive (1)'], digits=4))
print("═" * 60 + "\n")

⚙️ Generating TF-IDF Features for Lemmatized Process...
✅ Data Split: Training: 2400 | Testing: 600

🚀 Setting up Bayesian Optimization...
🧠 Commencing Search (Testing 10 different model architectures...)


Bayesian Tuning:   0%|          | 00:00 [? left]


🎯 Generating Predictions with the Champion Model...

════════════════════════════════════════════════════════════
 OPTIMIZED SVM EXPERIMENT: TF-IDF
════════════════════════════════════════════════════════════
BEST HYPERPARAMETERS FOUND:
 - C: 1.5870463392513299
 - gamma: 0.5703843027403095
 - kernel: linear
------------------------------
Final Accuracy: 82.17%
ROC-AUC Score:  0.8821

Detailed Classification Report (includes Precision/Recall/F1):
------------------------------
              precision    recall  f1-score   support

Negative (0)     0.8097    0.8395    0.8243       299
Positive (1)     0.8345    0.8040    0.8190       301

    accuracy                         0.8217       600
   macro avg     0.8221    0.8217    0.8216       600
weighted avg     0.8221    0.8217    0.8216       600

════════════════════════════════════════════════════════════



In [ ]:
import joblib

print(" Packaging files for deployment...")

# 1. Save the winning SVM model
# 'best_svm' is the variable saved from your Bayesian Search
joblib.dump(best_svm, 'champion_svm_model.pkl')

# 2. Save the TF-IDF Vectorizer
# The API needs this to translate user text into the exact same 3000 features!
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

print(" Success! Files saved:")
print(" - champion_svm_model.pkl")
print(" - tfidf_vectorizer.pkl")

 Packaging files for deployment...
 Success! Files saved:
 - champion_svm_model.pkl
 - tfidf_vectorizer.pkl


In [ ]:
from google.colab import files

print("📥 Initiating download of deployment files...")

# Download the Model
files.download('champion_svm_model.pkl')

# Download the Vectorizer
files.download('tfidf_vectorizer.pkl')

print("✅ Check your browser's download folder!")

In [ ]:
import os
import shutil
from google.colab import drive

print("🔗 Connecting to Google Drive...")
drive.mount('/content/drive')

# 2. Define the name of the new folder in your Drive
# You can change 'Sentiment_Project_Deployment' to whatever you want
drive_folder_path = '/content/drive/MyDrive/Sentiment_Project_Deployment'

print(f"\n📁 Creating folder: {drive_folder_path}")
# 3. Create the folder (exist_ok=True means it won't crash if the folder already exists)
os.makedirs(drive_folder_path, exist_ok=True)

# 4. Copy the files from Colab to your Google Drive
print("📦 Moving deployment files to Drive...")

# Copy Model
shutil.copy('champion_svm_model.pkl', f'{drive_folder_path}/champion_svm_model.pkl')
# Copy Vectorizer
shutil.copy('tfidf_vectorizer.pkl', f'{drive_folder_path}/tfidf_vectorizer.pkl')

print("\n✅ Success! Your files are safely stored in Google Drive.")
print("👉 You can now go to drive.google.com, right-click the folder, and share it with your friend!")

🔗 Connecting to Google Drive...
Mounted at /content/drive

📁 Creating folder: /content/drive/MyDrive/Sentiment_Project_Deployment
📦 Moving deployment files to Drive...

✅ Success! Your files are safely stored in Google Drive.
👉 You can now go to drive.google.com, right-click the folder, and share it with your friend!
